# Train baselines + a stronger model, save metrics + plots into reports/, and commit the results.

Set project root + imports

In [19]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

import matplotlib.pyplot as plt

# Make notebook run from project root
PROJECT_ROOT = Path.cwd().parent
os.chdir(PROJECT_ROOT)
print("CWD:", Path.cwd())

PROCESSED = Path("data/processed")
REPORTS = Path("reports")
FIGS = REPORTS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

CWD: c:\Users\josha\Codeacademy Projects


load the latest modeling table automatically

In [20]:
from pathlib import Path
import os

# Set project root explicitly based on the known folder name
PROJECT_ROOT = Path.cwd() / "Data Scientist Portfolio Project"
if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
else:
    # fallback: if you are already inside the project somewhere
    # (e.g., running from notebooks/), go up until we find the folder
    p = Path.cwd()
    while p != p.parent and (p / "data").exists() is False:
        p = p.parent
    os.chdir(p)

print("CWD set to:", Path.cwd())

CWD set to: c:\Users\josha\Codeacademy Projects\Data Scientist Portfolio Project


In [21]:
from pathlib import Path
import pandas as pd

PROCESSED = Path("data/processed")

# find the newest modeling parquet
candidates = sorted(PROCESSED.glob("modeling_*.parquet"), key=lambda p: p.stat().st_mtime, reverse=True)
assert candidates, f"No modeling parquet found in {PROCESSED.resolve()}"

model_path = candidates[0]
print("Using:", model_path.name)

df = pd.read_parquet(model_path)
df.shape, df.columns[:8]

Using: modeling_dinaciclib_auc.parquet


((476, 19195),
 Index(['depmap_id', 'auc', 'TSPAN6', 'TNMD', 'DPM1', 'SCYL3', 'C1orf112',
        'FGR'],
       dtype='str'))

In [22]:
from pathlib import Path

REPORTS = Path("reports")
FIGS = REPORTS / "figures"
FIGS.mkdir(parents=True, exist_ok=True)

print("Reports dir:", REPORTS.resolve())

Reports dir: C:\Users\josha\Codeacademy Projects\Data Scientist Portfolio Project\reports


split X/y + groups (leakage-safe by depmap_id)

In [23]:
# Target column
target_col = "auc"
assert target_col in df.columns, f"Expected '{target_col}' in modeling table."

# depmap_id as group key (in case duplicates exist)
assert "depmap_id" in df.columns, "Expected 'depmap_id' column."

y = df[target_col].astype(float)
X = df.drop(columns=[target_col])

groups = X["depmap_id"].astype(str).values
X = X.drop(columns=["depmap_id"])

print("X:", X.shape, "y:", y.shape, "unique groups:", len(np.unique(groups)))

X: (476, 19193) y: (476,) unique groups: 476


helper: select top-N genes by variance (keeps it fast)

In [24]:
class TopVarianceSelector(BaseEstimator, TransformerMixin):
    """Select top N features by variance (fit on train only)."""
    def __init__(self, top_n=2000):
        self.top_n = int(top_n)

    def fit(self, X, y=None):
        X_arr = X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)

        v = np.var(X_arr, axis=0)
        n = min(self.top_n, X_arr.shape[1])
        self.keep_idx_ = np.argsort(v)[::-1][:n]

        # keep feature names if X is a DataFrame
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        else:
            self.feature_names_in_ = None

        self.n_features_in_ = X_arr.shape[1]
        return self

    def transform(self, X):
        check_is_fitted(self, "keep_idx_")
        X_arr = X.to_numpy() if hasattr(X, "to_numpy") else np.asarray(X)
        return X_arr[:, self.keep_idx_]

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "keep_idx_")
        if input_features is None:
            if self.feature_names_in_ is None:
                return np.array([f"x{i}" for i in self.keep_idx_], dtype=object)
            input_features = self.feature_names_in_
        input_features = np.asarray(input_features, dtype=object)
        return input_features[self.keep_idx_]

train/val/test split (grouped)

In [25]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups[train_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)  # 0.1765 of train ≈ 0.15 of full
tr_idx, val_idx = next(gss2.split(X_train, y_train, groups=groups_train))

X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

print("Train:", X_tr.shape, "Val:", X_val.shape, "Test:", X_test.shape)

Train: (332, 19193) Val: (72, 19193) Test: (72, 19193)


metrics helper

In [26]:
def regression_metrics(y_true, y_pred):
    # RMSE that works even when mean_squared_error has no 'squared' kwarg
    rmse = float(mean_squared_error(y_true, y_pred) ** 0.5)
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))
    return {"rmse": rmse, "mae": mae, "r2": r2}

Baseline 1: Ridge (fast + strong baseline)

In [27]:
ridge_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("selector", TopVarianceSelector(top_n=2000)),
    ("scaler", StandardScaler()),
    ("model", RidgeCV(alphas=np.logspace(-3, 3, 25)))
])

ridge_pipe.fit(X_tr, y_tr)

pred_val = ridge_pipe.predict(X_val)
pred_test = ridge_pipe.predict(X_test)

ridge_val = regression_metrics(y_val, pred_val)
ridge_test = regression_metrics(y_test, pred_test)

ridge_val, ridge_test

({'rmse': 0.09999838557070788,
  'mae': 0.07993925076723753,
  'r2': -0.00664431646390784},
 {'rmse': 0.08444223669184049,
  'mae': 0.07044927146028682,
  'r2': 0.17478645092544032})

Baseline 2: ElasticNet (sparse-ish linear model)

In [28]:
enet_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("selector", TopVarianceSelector(top_n=2000)),
    ("scaler", StandardScaler()),
    ("model", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000, random_state=42))
])

enet_pipe.fit(X_tr, y_tr)

pred_val = enet_pipe.predict(X_val)
pred_test = enet_pipe.predict(X_test)

enet_val = regression_metrics(y_val, pred_val)
enet_test = regression_metrics(y_test, pred_test)

enet_val, enet_test

({'rmse': 0.095980747484446,
  'mae': 0.07941307166380529,
  'r2': 0.07261872140525083},
 {'rmse': 0.08711451443815962,
  'mae': 0.0706956487589069,
  'r2': 0.12173023138686145})

Model 3: XGBoost

In [29]:
from xgboost import XGBRegressor

# preprocessing -> numpy
prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("selector", TopVarianceSelector(top_n=2000)),
])

X_tr_p = prep.fit_transform(X_tr)
X_val_p = prep.transform(X_val)
X_test_p = prep.transform(X_test)

xgb = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=100,
)

xgb.fit(
    X_tr_p, y_tr.values,
    eval_set=[(X_val_p, y_val.values)],
    verbose=False
)

pred_val = xgb.predict(X_val_p)
pred_test = xgb.predict(X_test_p)

xgb_val = regression_metrics(y_val, pred_val)
xgb_test = regression_metrics(y_test, pred_test)

best_iter = getattr(xgb, "best_iteration", None)
xgb_val, xgb_test, best_iter

({'rmse': 0.09728282675652017,
  'mae': 0.07935780886058307,
  'r2': 0.047286253490178454},
 {'rmse': 0.08984071649923066,
  'mae': 0.0748989373128914,
  'r2': 0.06590013978345577},
 23)

save metrics + plots

In [30]:
results = {
    "dataset": model_path.name,
    "target": target_col,
    "n_train": int(len(X_tr)),
    "n_val": int(len(X_val)),
    "n_test": int(len(X_test)),
    "ridge": {"val": ridge_val, "test": ridge_test},
    "elasticnet": {"val": enet_val, "test": enet_test},
    "xgboost": {"val": xgb_val, "test": xgb_test, "best_iteration": int(xgb.best_iteration)},
}

metrics_path = REPORTS / f"metrics_{model_path.stem}.json"
with open(metrics_path, "w") as f:
    json.dump(results, f, indent=2)

print("Wrote:", metrics_path)

# Plot: Pred vs True (XGBoost on test)
plt.figure()
plt.scatter(y_test, pred_test, s=8)
plt.xlabel("True AUC")
plt.ylabel("Predicted AUC")
plt.title("XGBoost: Predicted vs True (Test)")
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
plt.plot(lims, lims)
fig_path = FIGS / f"xgb_pred_vs_true_{model_path.stem}.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.close()
print("Wrote:", fig_path)

Wrote: reports\metrics_modeling_dinaciclib_auc.json
Wrote: reports\figures\xgb_pred_vs_true_modeling_dinaciclib_auc.png


model comparison table (Ridge vs ElasticNet vs XGBoost)

In [31]:
summary = pd.DataFrame([
    {"model": "Ridge",      "split": "val",  **ridge_val},
    {"model": "Ridge",      "split": "test", **ridge_test},
    {"model": "ElasticNet", "split": "val",  **enet_val},
    {"model": "ElasticNet", "split": "test", **enet_test},
    {"model": "XGBoost",    "split": "val",  **xgb_val},
    {"model": "XGBoost",    "split": "test", **xgb_test},
])

# nice rounding for display
summary[["rmse","mae","r2"]] = summary[["rmse","mae","r2"]].round(4)
summary

,model,split,rmse,mae,r2
0,Ridge,val,0.1000,0.0799,-0.0066
1,Ridge,test,0.0844,0.0704,0.1748
2,ElasticNet,val,0.0960,0.0794,0.0726
3,ElasticNet,test,0.0871,0.0707,0.1217
4,XGBoost,val,0.0973,0.0794,0.0473
5,XGBoost,test,0.0898,0.0749,0.0659


Save comparison table as CSV

In [32]:
out_csv = Path("reports") / f"metrics_{model_path.stem}.csv"
summary.to_csv(out_csv, index=False)
print("Wrote:", out_csv)

Wrote: reports\metrics_modeling_dinaciclib_auc.csv
